In [1]:
import os
import pandas as pd
import polars as pl
import numpy as np
from sklearn.model_selection import train_test_split
from itertools import combinations

In [2]:
DATASET_PATH = '/group/pmc021/amunif/epi-thesis/workflow/11_HepG2_with_DeepChrome_preprocessing/dataset/'

# Generate the combination list

In [3]:
marker_list = ['H3K4me3', 'H3K9ac', 'H3K9me3', 'H3K27ac', 'H3K27me3']

In [4]:
all_combinations = [
    list(c)
    for r in range(1, len(marker_list) + 1)
    for c in combinations(marker_list, r)
]

In [5]:
print(all_combinations)

[['H3K4me3'], ['H3K9ac'], ['H3K9me3'], ['H3K27ac'], ['H3K27me3'], ['H3K4me3', 'H3K9ac'], ['H3K4me3', 'H3K9me3'], ['H3K4me3', 'H3K27ac'], ['H3K4me3', 'H3K27me3'], ['H3K9ac', 'H3K9me3'], ['H3K9ac', 'H3K27ac'], ['H3K9ac', 'H3K27me3'], ['H3K9me3', 'H3K27ac'], ['H3K9me3', 'H3K27me3'], ['H3K27ac', 'H3K27me3'], ['H3K4me3', 'H3K9ac', 'H3K9me3'], ['H3K4me3', 'H3K9ac', 'H3K27ac'], ['H3K4me3', 'H3K9ac', 'H3K27me3'], ['H3K4me3', 'H3K9me3', 'H3K27ac'], ['H3K4me3', 'H3K9me3', 'H3K27me3'], ['H3K4me3', 'H3K27ac', 'H3K27me3'], ['H3K9ac', 'H3K9me3', 'H3K27ac'], ['H3K9ac', 'H3K9me3', 'H3K27me3'], ['H3K9ac', 'H3K27ac', 'H3K27me3'], ['H3K9me3', 'H3K27ac', 'H3K27me3'], ['H3K4me3', 'H3K9ac', 'H3K9me3', 'H3K27ac'], ['H3K4me3', 'H3K9ac', 'H3K9me3', 'H3K27me3'], ['H3K4me3', 'H3K9ac', 'H3K27ac', 'H3K27me3'], ['H3K4me3', 'H3K9me3', 'H3K27ac', 'H3K27me3'], ['H3K9ac', 'H3K9me3', 'H3K27ac', 'H3K27me3'], ['H3K4me3', 'H3K9ac', 'H3K9me3', 'H3K27ac', 'H3K27me3']]


In [6]:
df = pl.DataFrame({"combination": all_combinations})

In [7]:
df

combination
list[str]
"[""H3K4me3""]"
"[""H3K9ac""]"
"[""H3K9me3""]"
"[""H3K27ac""]"
"[""H3K27me3""]"
…
"[""H3K4me3"", ""H3K9ac"", … ""H3K27me3""]"
"[""H3K4me3"", ""H3K9ac"", … ""H3K27me3""]"
"[""H3K4me3"", ""H3K9me3"", … ""H3K27me3""]"


In [8]:
df.write_parquet(os.path.join(DATASET_PATH, "marker_combinations.parquet"))

In [9]:
test_df = pl.read_parquet(os.path.join(DATASET_PATH, "marker_combinations.parquet"))
test_df

combination
list[str]
"[""H3K4me3""]"
"[""H3K9ac""]"
"[""H3K9me3""]"
"[""H3K27ac""]"
"[""H3K27me3""]"
…
"[""H3K4me3"", ""H3K9ac"", … ""H3K27me3""]"
"[""H3K4me3"", ""H3K9ac"", … ""H3K27me3""]"
"[""H3K4me3"", ""H3K9me3"", … ""H3K27me3""]"


# Generate train, validation, split set

In [10]:
# Load the dataset
all_features = pl.read_parquet(os.path.join(DATASET_PATH, 'HepG2_exp_histones.parquet'))
all_features

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len,value_1,value_2
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64,f64
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0,0.0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0,0.0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0888452,0.136965
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,4.04743,4.16567
"""XLOC_000008""","[0.0, 0.0, … 0.0]",6,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",12,100,"[0.0, 0.0, … 0.0]",4,100,"[0.0, 0.0, … 0.0]",0,100,26.7934,30.2747
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0,0.0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0,0.0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0,0.0


In [11]:
# Convert to numpy array
all_features_np = all_features.to_numpy()
print(all_features_np)
print(all_features_np.shape)

[['XLOC_000001'
  array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
  0 ... 100 0.0 0.0]
 ['XLOC_000003'
  array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
  0 ... 100 0.0 0.0]
 ['XLOC_000006'


In [12]:
# Split train, test, validation by index
data_indices = np.arange(len(all_features_np))
print(data_indices)

[    0     1     2 ... 22151 22152 22153]


In [13]:
# First split: 80% train, 20% temporary (for test + validation)
train_idx, temp_idx = train_test_split(
    data_indices, 
    test_size=0.2, 
    random_state=42  # For reproducibility
)

In [14]:
# Second split: Split temp_idx into 50% test and 50% validation
val_idx, test_idx = train_test_split(
    temp_idx, 
    test_size=0.5, 
    random_state=42  # Same random_state for consistency
)

In [15]:
print(train_idx)
print(val_idx)
print(test_idx)

[12829 12333  8225 ...  5390   860 15795]
[13202 15153  3113 ...  1158  6603 15289]
[17261  8344  1396 ...  7985   482 16493]


In [16]:
# Save train, val, and test into parquet file
train_idx_df = pd.DataFrame(train_idx, columns=['values'])
train_idx_df.to_parquet(os.path.join(DATASET_PATH, 'train_idx.parquet'))

val_idx_df = pd.DataFrame(val_idx, columns=['values'])
val_idx_df.to_parquet(os.path.join(DATASET_PATH, 'val_idx.parquet'))

test_idx_df = pd.DataFrame(test_idx, columns=['values'])
val_idx_df.to_parquet(os.path.join(DATASET_PATH, 'test_idx.parquet'))

In [17]:
train_idx_df = pd.read_parquet(os.path.join(DATASET_PATH, 'train_idx.parquet'))
len(train_idx_df["values"].to_list())

17723

In [18]:
val_idx_df = pd.read_parquet(os.path.join(DATASET_PATH, 'val_idx.parquet'))
len(val_idx_df["values"].to_list())

2215

In [19]:
test_idx_df = pd.read_parquet(os.path.join(DATASET_PATH, 'test_idx.parquet'))
len(test_idx_df["values"].to_list())

2215